# Week 8, Day 2: RAG prices for wrestling merch

**Lesson:** [week8/day2.ipynb](../../../week8/day2.ipynb), *RAG, the frontier agent and the ensemble agent*. **About 45 minutes.**

**What you practice here**

- Storing items and their prices in a Chroma collection, with vectors from a local encoder: `encoder.encode` and `collection.add`.
- Finding the nearest items with `collection.query`, and reading the first query's documents and metadatas.
- Putting the retrieved items and their prices into the prompt, and calling the frontier model through LiteLLM's `completion`, with the lesson's reasoning effort and seed.
- Blending the frontier model's estimate with the specialist's on Modal, as the lesson's ensemble does.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week8/day2", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson prices a product with RAG. It stores 400,000 products and their prices in Chroma, with vectors from a small
local encoder, finds the five products nearest to an item, and puts them with their prices into the prompt for
`gpt-5.1`. Then it blends that estimate with the fine-tuned specialist on Modal. Do the same for wrestling
merchandise: a store of items that sold recently, a search for the sales nearest to a new listing, the prompt, the
call, and the blend.

The parts that store, search and ask take Chroma's collection, the encoder, LiteLLM's `completion` or the Modal pricer
as parameters, and call them as the lesson does. The checks hand them stand-ins shaped like the real ones: the encoder
gives numpy vectors that are close for texts that share words, the collection ranks by distance as Chroma does, and
the model and the pricer answer with canned replies. Nothing is downloaded and nothing is spent.

## What each call should give

```python
add_sales(collection, encoder, SALES)    # one encoding of all the descriptions, one add: ids sale_0, sale_1, ...

find_similars(collection, encoder, "Undertaker action figure, 1991 Hasbro, complete with urn", n=2)
# [{'description': 'Undertaker action figure, 1991 Hasbro, with urn', 'category': 'Figures', 'price': 45.5},
#  {'description': 'Bret Hart action figure, Hasbro 1992, pink and black', 'category': 'Figures', 'price': 38.0}]

messages_for(LISTING, similars)
# [{'role': 'system', 'content': '... estimate the price ... reply with the price only ...'},
#  {'role': 'user', 'content': '... LISTING ... Undertaker action figure, 1991 Hasbro, with urn ... $45.50 ...'}]

estimate_price(completion, LISTING, similars)          # 52.0, from a reply such as '$52.00'
ensemble_price(completion, pricer, LISTING, similars)  # 0.8 * 52.0 + 0.2 * the specialist's price
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import re

### 1 · `add_sales(collection, encoder, sales)`

Stores sold items in a Chroma collection, each with its vector and its price. `sales` is a list of dicts with
`"description"`, `"category"` and `"price"`. Encode all the descriptions in one call to the encoder: it returns a numpy
array with one row per text, which the lesson turns into lists of floats before storing them. Then add the items to
the collection in one call: the descriptions as the documents, their vectors as the embeddings,
`{"category": ..., "price": ...}` as each one's metadata, and the ids `"sale_0"`, `"sale_1"`, ... numbered on from the
number of items that the collection already holds, so that a second call adds new rows instead of reusing ids. Return
nothing.

<details><summary>Hint</summary>

```python
start = collection.count()
vectors = encoder.encode(descriptions).astype(float).tolist()
collection.add(ids=ids, documents=descriptions, embeddings=vectors, metadatas=metadatas)
```

</details>

In [ ]:
def add_sales(collection, encoder, sales):
    # Write add_sales here, as described above.
    raise NotImplementedError("add_sales is not written yet")

In [ ]:
check("add_sales")

### 2 · `find_similars(collection, encoder, description, n=5)`

Finds the `n` stored sales nearest to a description, nearest first. Encode the description, given as a list of one
text, and query the collection once with that vector for `n` results. The results hold one list per query: read the
documents and the metadatas of the first (and only) query, as the lesson does. Return a list of dicts with the keys
of a sale: `{"description": ..., "category": ..., "price": ...}`.

<details><summary>Hint</summary>

```python
vector = encoder.encode([description]).astype(float).tolist()
results = collection.query(query_embeddings=vector, n_results=n)
results["documents"][0], results["metadatas"][0]
```

</details>

In [ ]:
def find_similars(collection, encoder, description, n=5):
    # Write find_similars here, as described above.
    raise NotImplementedError("find_similars is not written yet")

In [ ]:
check("find_similars")

### 3 · `messages_for(listing, similars)`

Builds the messages that ask for a listing's price, with similar sales as context. Unlike the lesson's single user
message, put the instructions in a system message of their own: it tells the model that it estimates the price of
wrestling merchandise and replies with the price only. Then a user message holds the listing and, for each of
`similars` (dicts as `find_similars` returns them), its description and its price with a dollar sign and two
decimals, such as `$45.50`, as the lesson's `make_context` writes them. Return the two messages, the system message
first.

<details><summary>Hint</summary>

```python
f"{similar['description']}\nSold for ${similar['price']:.2f}"
[{"role": "system", "content": ...}, {"role": "user", "content": ...}]
```

</details>

In [ ]:
def messages_for(listing, similars):
    # Write messages_for here, as described above.
    raise NotImplementedError("messages_for is not written yet")

In [ ]:
check("messages_for")

### 4 · `estimate_price(completion, listing, similars, model="gpt-5.1")`

Asks the frontier model for the listing's price, with the similar sales as context, and returns the number.
`completion` is LiteLLM's `completion` function. Make one call to it with the model `model`, the messages from
`messages_for(listing, similars)`, the reasoning effort `"none"` and the seed 42, as the lesson does, and read the
text of the reply. Return the first number in that text as a float, ignoring `$` and `,`, as the lesson's `get_price`
does: `"$1,299.99"` gives `1299.99`. A reply with no number gives `0.0`.

<details><summary>Hint</summary>

```python
response = completion(model=model, messages=..., reasoning_effort="none", seed=42)
reply = response.choices[0].message.content.replace("$", "").replace(",", "")
match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
```

</details>

In [ ]:
def estimate_price(completion, listing, similars, model="gpt-5.1"):
    # Write estimate_price here, as described above.
    raise NotImplementedError("estimate_price is not written yet")

In [ ]:
check("estimate_price")

### 5 · `ensemble_price(completion, pricer, listing, similars, weights=(0.8, 0.2))`

Blends the frontier model's estimate with the specialist's on Modal, and returns the weighted sum. Get the frontier's
price with `estimate_price(completion, listing, similars)`, and the specialist's by calling the pricer's `price` method
remotely, once, with the listing. `pricer` is the deployed pricer's instance, as in the lesson and Day 1. Return
`weights[0] * frontier + weights[1] * specialist` as a float. The lesson's three-way blend gives the frontier 0.8;
here the specialist takes the other 0.2.

<details><summary>Hint</summary>

```python
specialist = pricer.price.remote(listing)
```

</details>

In [ ]:
def ensemble_price(completion, pricer, listing, similars, weights=(0.8, 0.2)):
    # Write ensemble_price here, as described above.
    raise NotImplementedError("ensemble_price is not written yet")

In [ ]:
check("ensemble_price")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 8, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

The encoder runs on your machine for free, after a download of about 90 MB the first time. Each estimate from
`gpt-5.1` costs a fraction of a cent, and the specialist on Modal bills the seconds its container runs.

1. Load your keys with `load_dotenv(override=True)` and make
   `encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")`. Open a collection of your own in a
   temporary folder, `chromadb.PersistentClient(path=tempfile.mkdtemp())`, then `get_or_create_collection("merch_sales")`,
   and fill it with the `SALES` in the cell below, using `add_sales`.
2. For `LISTING`, print `find_similars(...)`, the user message of `messages_for(...)`, and
   `estimate_price(completion, ...)`, with `from litellm import completion`.
3. Ask the lesson's own store about the same listing (the store you built in the lesson; skip this step if you did not
   build it): open `chromadb.PersistentClient(path=str(ROOT / "week8" / "products_vectorstore"))` and take
   `get_collection("products")`, which fails if the store is not there. Print its `count()`, then compare the
   estimate with step 2's. Its metadata hold `category` and `price` too.
4. Blend with the specialist you deployed on Day 1: after `import modal`, print
   `ensemble_price(completion, modal.Cls.from_name("pricer-service", "Pricer")(), LISTING, similars)`.

<details><summary>Stuck? One way to write it</summary>

```python
import tempfile

import chromadb
from dotenv import load_dotenv
from litellm import completion
from sentence_transformers import SentenceTransformer

SALES = [
    {"description": "Undertaker action figure, 1991 Hasbro, with urn", "category": "Figures", "price": 45.5},
    {"description": "Bret Hart action figure, Hasbro 1992, pink and black", "category": "Figures", "price": 38.0},
    {"description": "Stone Cold Steve Austin action figure, Elite series, with vest and belt", "category": "Figures",
     "price": 24.99},
    {"description": "Replica Winged Eagle championship title belt, adult size", "category": "Belts", "price": 389.0},
    {"description": "Replica Big Gold championship belt, adult size, zinc plates", "category": "Belts", "price": 420.0},
    {"description": "WrestleMania III event program, 1987, Pontiac Silverdome", "category": "Programs", "price": 60.0},
    {"description": "Hulk Hogan Hulkamania t-shirt, yellow, size L", "category": "Clothing", "price": 18.0},
    {"description": "Macho Man Randy Savage replica sunglasses, pink", "category": "Accessories", "price": 22.5},
]
LISTING = "Undertaker action figure, 1991 Hasbro, complete with urn, light wear"

load_dotenv(override=True)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
merch = chromadb.PersistentClient(path=tempfile.mkdtemp()).get_or_create_collection("merch_sales")
add_sales(merch, encoder, SALES)

similars = find_similars(merch, encoder, LISTING)
for sale in similars:
    print(sale)
print(messages_for(LISTING, similars)[1]["content"])
print("With merch sales:", estimate_price(completion, LISTING, similars))
```

```python
store = chromadb.PersistentClient(path=str(ROOT / "week8" / "products_vectorstore"))
products = store.get_collection("products")  # raises if the lesson's store is not there
print(products.count(), "products in the lesson's store")
similars = find_similars(products, encoder, LISTING)
print("With Amazon products:", estimate_price(completion, LISTING, similars))
```

```python
import modal

pricer = modal.Cls.from_name("pricer-service", "Pricer")()
print("Blended with the specialist:", ensemble_price(completion, pricer, LISTING, similars))
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.
# Sales that sold recently, and a new listing to price:
SALES = [
    {"description": "Undertaker action figure, 1991 Hasbro, with urn", "category": "Figures", "price": 45.5},
    {"description": "Bret Hart action figure, Hasbro 1992, pink and black", "category": "Figures", "price": 38.0},
    {"description": "Stone Cold Steve Austin action figure, Elite series, with vest and belt", "category": "Figures",
     "price": 24.99},
    {"description": "Replica Winged Eagle championship title belt, adult size", "category": "Belts", "price": 389.0},
    {"description": "Replica Big Gold championship belt, adult size, zinc plates", "category": "Belts", "price": 420.0},
    {"description": "WrestleMania III event program, 1987, Pontiac Silverdome", "category": "Programs", "price": 60.0},
    {"description": "Hulk Hogan Hulkamania t-shirt, yellow, size L", "category": "Clothing", "price": 18.0},
    {"description": "Macho Man Randy Savage replica sunglasses, pink", "category": "Accessories", "price": 22.5},
]
LISTING = "Undertaker action figure, 1991 Hasbro, complete with urn, light wear"